In [7]:
import ee
import time

In [8]:
reach_id_field = "reach_id"
min_grwl_pixels = 100 # Note that this translates into 900 Sentinel-2 pixels

In [9]:
ee.Initialize(project="lmb-river-color")
# load in earth engine
# water_ee = ee.FeatureCollection("projects/lmb-river-color/assets/water_poly");
# load locally
# water_local = gpd.read_file('data/water_polygons.gpkg')
# water_ee = ee.FeatureCollection("projects/lmb-river-color/assets/water_polygons_conus_buffered")
water_ee = ee.FeatureCollection("projects/lmb-river-color/assets/water_polygons_buffered")
water_ee = water_ee.map(lambda f: f.simplify(10))
grwl_mask = ee.ImageCollection("projects/sat-io/open-datasets/GRWL/water_mask_v01_01")
grwl_mask_img = grwl_mask.mosaic()


In [10]:
water_mask = grwl_mask_img.gt(0)
pixel_counts = water_mask.reduceRegions(
    collection=water_ee,
    reducer=ee.Reducer.sum().setOutputs(['grwl_pixel_count']),
    scale=30
)

In [11]:
water_ee_filtered = pixel_counts.filter(
    ee.Filter.gte("grwl_pixel_count", min_grwl_pixels)
)

In [12]:
# Get majority water body type from GRWL
water_ee_filtered = grwl_mask_img.select('b1').reduceRegions(
    collection = water_ee_filtered,
    reducer = ee.Reducer.mode().setOutputs(['grwl_code']),
    scale=30
)

In [13]:
s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2017-01-01', '2026-12-31')
    .filterBounds(water_ee.geometry())
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
    #.map(lambda i: i.divide(10000))
)

# include only water pixels
s2 = s2.map(lambda i: i.addBands(i.select('SCL').eq(6).rename('water')) )
s2 = s2.map(lambda i: i.updateMask(i.select('water').eq(1).focalMin(2)) ) # focalMin to remove shoreline pixels
#Map.add_layer(s2.filterDate('2023-11-21', '2023-11-22'), visualization, 'RGB - water')


In [14]:
# dlDir = 'G:/My Drive/reach_data_sentinel' 
# filesDown = np.array(os.listdir(dlDir))
# filesDown = filesDown[['.csv' in f for f in filesDown]]
# filesDown = [int(i.replace(".csv", "")) for i in filesDown]

# reach_id  = [i for i in water_local.reach_id if i not in filesDown]

# print(len(reach_id))

In [15]:
# Get median by reach for each image
def calc_stat(image, reach_feature):
    medians = image.select(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A',]).reduceRegion(
        geometry=reach_feature.geometry(),
        reducer=ee.Reducer.median(),
        scale=10  # meters
    )

    pixel_count = image.select('B1').reduceRegion(
        geometry=reach_feature.geometry(),
        reducer=ee.Reducer.count(), 
        scale=10)

    output = ee.Feature(None, medians)\
        .set({'date': ee.Date(image.get('system:time_start'))})\
        .set({"reach_id": reach_feature.get(reach_id_field)})\
        .set({'azimuth': image.get('MEAN_SOLAR_AZIMUTH_ANGLE')})\
        .set({'zenith': image.get('MEAN_SOLAR_ZENITH_ANGLE')})\
        .set({"pixelCount": pixel_count.get("B1")})\
        .set({'tile': image.get('MGRS_TILE')})\
        .set({'cloud_cover': image.get('CLOUDY_PIXEL_PERCENTAGE')})\
        .set({'grwl_pixel_count': reach_feature.get('sum')})\
        .set({'grwl_code': reach_feature.get('grwl_code')})

    
    return output


def calc_stat_chunk(image, reach_features):
    bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A']

    medians = image.select(bands).reduceRegions(
        collection=reach_features,
        reducer=ee.Reducer.median(),
        scale=10
    )
    # Add pixel count.  
    # The medians object has the same geometry as reach_features. 
    medians = image.select('B1').reduceRegions(
        collection=medians,
        reducer=ee.Reducer.count().setOutputs(['pixel_count']), 
        scale=10)
    
    return medians.map(lambda f: f.set({
        'date': ee.Date(image.get('system:time_start')),
        'azimuth': image.get('MEAN_SOLAR_AZIMUTH_ANGLE'),
        'zenith': image.get('MEAN_SOLAR_ZENITH_ANGLE'),
        'tile': image.get('MGRS_TILE'),
        'cloud_cover': image.get('CLOUDY_PIXEL_PERCENTAGE'),
        'pixel_count': f.get('pixel_count'),
        'grwl_pixel_count': f.get('grwl_pixel_count'),
        'grwl_code': f.get('grwl_code')
    }))



    

# Taken from https://github.com/johngardner87/riverSR/blob/4a55d5247a269e9124eba6747df71f80c157a899/src/GEE_pull_functions_rivers.py#L303C1-L322C11
def maximum_no_of_tasks(MaxNActive, waitingPeriod):
  ##maintain a maximum number of active tasks
  time.sleep(10)
  ## initialize submitting jobs
  ts = list(ee.batch.Task.list())

  NActive = 0
  for task in ts:
       if ('RUNNING' in str(task) or 'READY' in str(task)):
           NActive += 1
  ## wait if the number of current active tasks reach the maximum number
  ## defined in MaxNActive
  while (NActive >= MaxNActive):
      time.sleep(waitingPeriod) # if over maximum no. of active tasks, wait for waitingPeriod and check again
      ts = list(ee.batch.Task.list())
      NActive = 0
      for task in ts:
        if ('RUNNING' in str(task) or 'READY' in str(task)):
          NActive += 1
  return()


In [16]:
# #  TESTING WITH ONE REACH FIRST
# #  get array of reach_id from the reach_id field in the water_ee feature collection
# reach_id  = water_ee.aggregate_array(reach_id_field).getInfo()
# ids = reach_id[0:3]
# i = 0
# reach_features = water_ee_filtered.filter(ee.Filter.inList(reach_id_field, ids))
# s2_reach = s2.filterBounds(reach_features)
# median = ee.FeatureCollection(
#     s2_reach.map(lambda img: calc_stat_chunk(img, reach_features))
# ).flatten()
# dataOut = ee.batch.Export.table.toDrive(collection = median,
#                                         description = f'chunk_{i}_myreaches_with_count',
#                                         selectors=[
#                                             reach_id_field,
#                                             'date',
#                                             'B1','B2','B3','B4',
#                                             'B5','B6','B7','B8','B8A',
#                                             'pixel_count',
#                                             'azimuth','zenith','tile','cloud_cover', 
#                                             'grwl_pixel_count','grwl_code'
#                                         ],
#                                         folder = 'reach_data_sentinel',
#                                         fileFormat = 'CSV')
# # only allow 30 active tasks at a time, wait 60 seconds if there are already 30 active tasks
# maximum_no_of_tasks(30, 60)
# dataOut.start()

In [17]:
def chunker(seq, size):
    return ((seq[pos:pos + size], int(pos/size)) for pos in range(0, len(seq), size))

In [18]:
# There are about 8000 reaches total, 
# so we will chunk into groups of 50 reaches, which should be about 160 jobs total
reach_id  = water_ee_filtered.aggregate_array(reach_id_field).getInfo()
for (ids, i) in chunker(reach_id, 50):
    reach_features = water_ee_filtered.filter(ee.Filter.inList(reach_id_field, ids))
    s2_reach = s2.filterBounds(reach_features)
    median = ee.FeatureCollection(
        s2_reach.map(lambda img: calc_stat_chunk(img, reach_features))
    ).flatten()
    dataOut = ee.batch.Export.table.toDrive(collection = median,
                                            description = f'chunk_{i}_myreaches_with_count',
                                            selectors=[
                                                reach_id_field,
                                                'date',
                                                'B1','B2','B3','B4',
                                                'B5','B6','B7','B8','B8A',
                                                'pixel_count',
                                                'azimuth','zenith','tile','cloud_cover', 
                                                'grwl_pixel_count','grwl_code'
                                            ],
                                            folder = 'reach_data_sentinel',
                                            fileFormat = 'CSV')
    # only allow 15 active tasks at a time, wait 60 seconds if there are already 15 active tasks
    maximum_no_of_tasks(15, 60)
    dataOut.start()

In [19]:
# docs on ee.batch https://gee-python-api.readthedocs.io/en/latest/ee.html#ee.batch.Task.active
# View tasks GUI here: https://code.earthengine.google.com/tasks
ts = list(ee.batch.Task.list())
ts

[<Task EPXT2JFDJ3WQ77ANIEM46YCO EXPORT_FEATURES: chunk_165_myreaches_with_count (READY)>,
 <Task KGXJOBE5HHRCG4X5VYTOMEAH EXPORT_FEATURES: chunk_164_myreaches_with_count (READY)>,
 <Task F4E5WNFIBLNRNTGTRCVW26MK EXPORT_FEATURES: chunk_163_myreaches_with_count (READY)>,
 <Task H2QM5AAV7V34CICRDUF7WKHJ EXPORT_FEATURES: chunk_162_myreaches_with_count (READY)>,
 <Task OKUVHUPZUZW6W7UQNCPOSUAY EXPORT_FEATURES: chunk_161_myreaches_with_count (READY)>,
 <Task WKEE25VZ2CYQUFAO4BY3SPTM EXPORT_FEATURES: chunk_160_myreaches_with_count (READY)>,
 <Task A5NNOFHKNK7DBCWODDQ2JUGI EXPORT_FEATURES: chunk_159_myreaches_with_count (READY)>,
 <Task 5IWO7RA7AUJCXM4OEJ7WTE3A EXPORT_FEATURES: chunk_158_myreaches_with_count (READY)>,
 <Task 2FUWBLKABV6F7KUWJ2HA7353 EXPORT_FEATURES: chunk_157_myreaches_with_count (READY)>,
 <Task 2P5RTMRG6COURBXYZUL5WLQK EXPORT_FEATURES: chunk_156_myreaches_with_count (READY)>,
 <Task VDBL7ZZ5KXVRPK44EPHNJNG2 EXPORT_FEATURES: chunk_155_myreaches_with_count (READY)>,
 <Task 4GN